# Lesson 5

This notebook builds from simple prompt improvements to reusable prompt templates in `prompts.py`.

## Video 1 — Why Prompt Wording Matters

In [1]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import the Anthropic SDK
import anthropic

# Create the client
client = anthropic.Anthropic()

# Model used in this lesson
MODEL = "claude-sonnet-5-5"

### vague vs specific

In [2]:
# A vague prompt leaves many possible directions open
vague_prompt = "Tell me about dogs."

resp = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": vague_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

Claude: # Dogs (*Canis lupus familiaris*)

## Origins
Dogs were the first animal domesticated by humans, at least 15,000 years ago and possibly much earlier. They descend from an extinct population of wolves. The exact when and where is still debated, but the relationship likely began when wolves scavenged near human camps and gradually became companions and helpers.

## Diversity
Dogs show more variation in size and shape than any other mammal. A Chihu


In [4]:
# A more specific prompt narrows the task and output
specific_prompt = """
List 3 dog breeds that are suitable for apartment living.

For each breed:
- Give the breed name
- Give one short reason

Keep the answer concise.
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": specific_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

1. **French Bulldog**: Small and calm, with low exercise needs and minimal barking.
2. **Cavalier King Charles Spaniel**: Gentle and adaptable, and happy with moderate walks and indoor play.
3. **Shih Tzu**: Compact and affectionate, and content with short walks and lounging indoors.


## Video 2 — Four Core Prompting Techniques

### Build a Strong Prompt Step by Step

In [5]:
# Sample context for the next examples
ticket_text = """
I upgraded my subscription yesterday, but my account still shows the free plan.
I have already logged out and back in twice.
"""

In [6]:
# 1. Clear instruction
prompt = "Summarise the support ticket in one sentence."

print(prompt)

Summarise the support ticket in one sentence.


In [7]:
# 2. Add a role
prompt = """
You are a support agent.

Summarise the support ticket in one sentence.
"""

print(prompt)


You are a support agent.

Summarise the support ticket in one sentence.



In [8]:
# 3. Add the context and separate it with tags
prompt = f"""
You are a support agent.

Summarise the support ticket in one sentence.

<ticket>
{ticket_text}
</ticket>
"""

print(prompt)


You are a support agent.

Summarise the support ticket in one sentence.

<ticket>

I upgraded my subscription yesterday, but my account still shows the free plan.
I have already logged out and back in twice.

</ticket>



In [9]:
# 4. Specify the output format
prompt = f"""
You are a support agent.

Summarise the support ticket in one sentence.

<ticket>
{ticket_text}
</ticket>

Return only the summary.
"""

print(prompt)


You are a support agent.

Summarise the support ticket in one sentence.

<ticket>

I upgraded my subscription yesterday, but my account still shows the free plan.
I have already logged out and back in twice.

</ticket>

Return only the summary.



In [10]:
# Send the engineered prompt to Claude
resp = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

Claude: The customer upgraded their subscription yesterday, but their account still shows the free plan even after logging out and back in twice.


## Video 3 — Few-Shot Prompting

### Zero-Shot First

In [11]:
# Input we want to classify
text = "The update fixed the issue and everything works now."

# Zero-shot: instruction only
zero_shot_prompt = f"""
Classify the sentiment as positive or negative.

Text: "{text}"
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=50,
    messages=[
        {"role": "user", "content": zero_shot_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

Claude: **Positive**

The text describes a problem being resolved ("fixed the issue") and a satisfactory outcome ("everything works now"), both of which express a fav


### Few shot prompting

In [16]:
text = "The update didn't fix the issue and I'm not happy"

In [17]:
# Few-shot: show examples of the pattern we want
few_shot_prompt = f"""
Classify the sentiment.

Text: "Great job!" -> positive
Text: "This is broken." -> negative
Text: "The service was excellent." -> positive

Text: "{text}" ->
"""

resp = client.messages.create(
    model=MODEL,
    max_tokens=50,
    messages=[
        {"role": "user", "content": few_shot_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

negative


## Video 4 — Reusable Prompt Templates

In [18]:
# Import the reusable templates from prompts.py

from prompts import summarize_prompt, answer_with_context

In [19]:
# Build a prompt from the basic template
sample_text = """
The engineering team completed the migration on Friday.
The old database has now been placed in read-only mode.
"""

summary_prompt = summarize_prompt(sample_text)

# Inspect the final prompt generated by the function
print(summary_prompt)

You are a concise editor.

Summarise the text below in one clear sentence.

<text>

The engineering team completed the migration on Friday.
The old database has now been placed in read-only mode.
 
</text>

Return only the summary.


In [21]:
# Send the generated prompt
resp = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {"role": "user", "content": summary_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

The engineering team finished the migration on Friday, and the old database is now read-only.


In [22]:
# Context and question for the advanced template
context = """
The Premium plan includes 100 GB of storage and priority email support.
The Standard plan includes 25 GB of storage and standard email support.
Only the Premium plan includes access to the analytics dashboard.
"""

question = "Which plan includes the analytics dashboard?"

# Build the prompt from prompts.py
context_prompt = answer_with_context(question, context)

# Inspect the final prompt
print(context_prompt)

You are a careful research assistant.

Answer the question using only the context provided below.

<context>

The Premium plan includes 100 GB of storage and priority email support.
The Standard plan includes 25 GB of storage and standard email support.
Only the Premium plan includes access to the analytics dashboard.

</context>

<question>
Which plan includes the analytics dashboard?
</question>

Requirements:
- Use only information supported by the context.
- If the context does not contain enough information, say that clearly.
- Keep the answer concise and directly relevant.

Return only the final answer.


In [23]:
# Send the advanced prompt
resp = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user", "content": context_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

Claude: The Premium plan includes the analytics dashboard. According to the context, it is the only plan that does.


In [24]:
# Test the guardrail in the advanced prompt
missing_question = "How much does the Premium plan cost?"

missing_context_prompt = answer_with_context(
    missing_question,
    context
)

resp = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user", "content": missing_context_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

Claude: The context does not state the price of the Premium plan. It only says that the plan includes 100 GB of storage, priority email support, and access to the analytics dashboard.
